In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import xgboost as xgb
import lightgbm as lgb

df = pd.read_csv("netflix_engineered.csv")

In [7]:
# IMPORTANT: Primary_Genre and Duration_Value/Unit are deliberately excluded from the feature set.
# Netflix labels genres differently by type (e.g. "Crime TV Shows" only ever appears on TV Shows,
# "Action & Adventure" only on Movies) and duration units are Movie=minutes/TV=seasons by definition -
# both would leak the answer directly rather than reflect a model that actually learned a pattern.

feature_num = ['release_year', 'Year_Added', 'Month_Added', 'Cast_Count', 'Genre_Count', 'Title_Length',
               'Description_WordCount', 'Has_Director', 'Has_Country']
feature_cat = ['rating', 'Primary_Country']

X = df[feature_num + feature_cat]
y = (df['type'] == 'TV Show').astype(int) # 1 = TV Show, 0 = Movie

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify=y)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), feature_num),
    ('cat', OneHotEncoder(handle_unknown = 'ignore'), feature_cat)
])

In [10]:
models = {
    "XGBoost": xgb.XGBClassifier(n_estimators = 200, random_state = 42, eval_metric ='logloss'),
    "LightGBM": lgb.LGBMClassifier(n_estimators = 200, random_state = 42, verbose = -1),
    "Neural Network (MLP)": MLPClassifier(hidden_layer_sizes =(64, 32), max_iter = 500, random_state = 42)
}

In [14]:
results = {}
for name, model in models.items():
    pipe = Pipeline([('pre', preprocessor), ('clf', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results[name] = {
        "Accuracy": accuracy_score(y_test,pred),
        "Precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
    }
    print(f" -- {name} --")
    print(classification_report(y_test, pred, target_names = ['Movie', 'TV Show']))
    
    

 -- XGBoost --
              precision    recall  f1-score   support

       Movie       1.00      1.00      1.00      1226
     TV Show       0.99      0.99      0.99       535

    accuracy                           1.00      1761
   macro avg       0.99      0.99      0.99      1761
weighted avg       1.00      1.00      1.00      1761



C:\Users\sholy\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


 -- LightGBM --
              precision    recall  f1-score   support

       Movie       1.00      1.00      1.00      1226
     TV Show       0.99      0.99      0.99       535

    accuracy                           1.00      1761
   macro avg       1.00      1.00      1.00      1761
weighted avg       1.00      1.00      1.00      1761

 -- Neural Network (MLP) --
              precision    recall  f1-score   support

       Movie       0.98      0.97      0.97      1226
     TV Show       0.93      0.95      0.94       535

    accuracy                           0.96      1761
   macro avg       0.96      0.96      0.96      1761
weighted avg       0.96      0.96      0.96      1761



In [15]:
results_df = pd.DataFrame(results).T
results_df.to_csv("classification_report.csv")
print(results_df)

                      Accuracy  Precision    recall        F1
XGBoost               0.995457   0.992523  0.992523  0.992523
LightGBM              0.996025   0.992537  0.994393  0.993464
Neural Network (MLP)  0.964225   0.933824  0.949533  0.941613
